# Rung 1 — the content-based induction task (issue #200)

One Kaggle/Colab session, decided **before** seeing results. The
pre-registration is `07_capstone/notes/variable-distance-induction-preregistration.md`
and the design is ADR-0033.

On the periodic task the previous occurrence of a token is always at `q-L`,
so "attend to `q-L+1`" and "attend to the token after the previous
occurrence" are the *same function* on the training distribution, and the
2026-10-19 probe showed the model had learned the offset. This task puts
the repeats at *varying* distances, so the fixed-offset shortcut does not
exist: the best single fixed offset explains 17.6% of the copy queries
(measured on the data before this run), while a content-matching head can
reach 100%.

| Step | What runs | Why |
|------|-----------|-----|
| 0 | **Pilot**: 20 epochs of the content config, timed | Measures this GPU's real throughput (and the fresh-batch data generation, which is pure Python) so the session budget is planned from a measurement instead of an estimate |
| 1 | `exp1 --task variable --standard --seeds 0,1,2` with `--stop-on-confirmed --stop-rule mechanism` | The **emergence** measurement: when does the two-step circuit appear? Ends each seed as soon as the detector confirms a head above the first layer, twice in a row |
| 2 | the same, no early stop | The **full-budget** measurement: 3000 epochs (384k optimizer steps per seed). The model it leaves behind is the one Rungs 4 and 5 analyse |
| 3 | `exp4 --from-checkpoint <exp1's checkpoint dir> --source-task variable` | Activation patching + head ablation + path patching **on the very model Rung 1 trained**, measured at a *copy query* |
| 4 | `exp5 --activations-from <rolling checkpoint> --source-task variable` | The SAE on real activations from the same model |

**The verdict is computed by code, not by hand.** Step 2b checks the seven
pre-registered criteria and prints the verdict per seed. A seed is never
dropped and the run is never repeated "to get a better label".


In [ ]:
BRANCH = "main"  # a pushed branch or tag — pin it so the manifest's git_sha is a real commit
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo
!git rev-parse --short HEAD

In [ ]:
# The GPU images already ship the CUDA torch stack plus numpy, matplotlib, tqdm and
# pyyaml. `uv sync --frozen` would re-resolve the ~5GB CUDA wheel set into the
# notebook's limited working disk and die mid-download — exactly how the first
# grokking kernel errored (2026-10-14). Use the preinstalled stack and fill only a
# real gap; the manifests record the torch/python versions actually used.
import importlib.util
import sys

missing = [
    m
    for m in ("torch", "numpy", "matplotlib", "tqdm", "yaml")
    if importlib.util.find_spec(m) is None
]
if missing:
    !pip install -q {' '.join(missing)}

import torch

assert torch.cuda.is_available(), "no GPU — enable one before running"
print(torch.__version__, "|", torch.cuda.get_device_name(0), "| py", sys.version.split()[0])

## Step 0 — Pilot: measure this GPU before spending the session

Twenty epochs of the **content config** (same shape, same fresh-batch
regime, same detection cadence), timed. The output is the number the
budget has to be planned from: the old notebook estimated "~2h" per run,
which was never measured.

If the projection does not fit the session, the fix is **one seed per
session** (set `SEEDS` below to a single seed), never a shorter budget: a
truncated run is exactly how this repository produced under-trained false
negatives.


In [ ]:
# 20 epochs of the standard config: 20 * 128 = 2560 optimizer steps.
import os, time
t0 = time.time()
!python -m src.experiments.exp1_induction_heads --task variable --vocab-size 2048 --seq-len 64 --d-model 64 --n-layers 2 --n-heads 4 --epochs 20 --batch-size 64 --num-train 8192 --batches fresh --seeds 0 --detect-every 10 --manifest-path results/probe_pilot_content.json
pilot_seconds = time.time() - t0
steps = 20 * 128
rate = steps / max(pilot_seconds, 1e-6)
full = 3000 * 128
print(f"pilot: {pilot_seconds:.1f}s for {steps} steps -> {rate:.1f} steps/s")
print(f"projected standard run: {full / max(rate, 1e-6) / 60:.1f} min per seed")
print(f"projected 3 seeds:       {3 * full / max(rate, 1e-6) / 3600:.2f} h")

## Step 1 — The emergence measurement

How early does the two-step circuit appear? `--stop-on-confirmed
--stop-rule mechanism` ends each seed as soon as the detector confirms a
head **above the first layer** (the two-step circuit: a duplicate-token
head plus a K-composed copy head) at two consecutive checks.

`SEEDS` is an environment variable so a short session can run one seed at
a time; the checkpoints and the per-seed manifest are the same either way.


In [ ]:
import os
os.environ["SEEDS"] = "0,1,2"  # set to a single seed (e.g. "0") if the session is short
!python -m src.experiments.exp1_induction_heads --task variable --standard --batches fresh --seeds $SEEDS --checkpoint-dir checkpoints/induction_content_emergence --checkpoint-every 250 --resume --detect-every 50 --stop-on-confirmed --stop-rule mechanism --manifest-path results/induction_content_emergence.json

## Step 2 — The full-budget run (the measurement Rungs 4 and 5 use)

The canonical config the pre-registration names, at its full budget: 3000
epochs = 384k optimizer steps per seed, no early stop. The model this run
leaves behind is the one the circuit-patching and SAE rungs analyse.

Why a second run and not just step 1: an early stop ends the run at the
*first* confirmation of the mechanism, which is a different question from
"what does the model settle into at the full budget".


In [ ]:
!python -m src.experiments.exp1_induction_heads --task variable --standard --batches fresh --seeds $SEEDS --checkpoint-dir checkpoints/induction_content --checkpoint-every 250 --resume --detect-every 50 --manifest-path results/induction_content.json

## Step 2b — The verdict, computed from the manifest

The seven pre-registered criteria, in the order they were frozen. A
criterion is met when the *seed* meets it; the run is POSITIVE when a
majority of seeds meet all seven, PARTIAL when a head is confirmed but
the causal criteria fail, and NEGATIVE otherwise.

The criteria are read from the manifest, so the verdict cannot drift from
the measurement: if the code below and the pre-registration disagree, the
test suite says so.


In [ ]:
import json

CRITERIA = [
    ("content mass",            "peak_content_mass",        lambda m: m["peak_content_mass"] >= 0.30),
    ("content beats position",  "peak_offset_mass",         lambda m: m["peak_content_mass"] - m["peak_offset_mass"] >= 0.10),
    ("copy accuracy",           "copy_accuracy",            lambda m: m["copy_accuracy"] >= 0.60),
    ("non-modal accuracy",      "copy_accuracy_nonmodal",   lambda m: m["copy_accuracy_nonmodal"] >= 0.50),
    ("beats fixed offset",      "fixed_offset_baseline",    lambda m: m["copy_accuracy"] - m["fixed_offset_baseline"] >= 0.20),
    ("ablation drop",           "mean_copy_ablation_drop",  lambda m: m["mean_copy_ablation_drop"] >= 0.10),
    ("path patching",           "mean_path_patch_effect",   lambda m: m["mean_path_patch_effect"] >= 0.20),
]


def verdict(metrics):
    checks = {name: bool(fn(metrics)) for name, _key, fn in CRITERIA}
    confirmed = int(metrics["total_induction_heads"]) > 0
    if all(checks.values()):
        return "POSITIVE", checks
    if confirmed:
        return "PARTIAL", checks
    return "NEGATIVE", checks


def report(path, label):
    m = json.load(open(path))
    print(f"\n=== {label} ===")
    for seed, metrics in zip(m["seeds"], m["per_seed_metrics"]):
        v, checks = verdict(metrics)
        print(
            f"seed {seed}: {v} | heads={int(metrics['total_induction_heads'])} "
            f"(L0 {int(metrics['confirmed_heads_first_layer'])}, "
            f"later {int(metrics['confirmed_heads_later_layers'])}) | "
            f"copy acc {metrics['copy_accuracy']:.4f} "
            f"(non-modal {metrics['copy_accuracy_nonmodal']:.4f}, "
            f"fixed-offset baseline {metrics['fixed_offset_baseline']:.4f}) | "
            f"content mass {metrics['peak_content_mass']:.3f} vs "
            f"offset mass {metrics['peak_offset_mass']:.3f} | "
            f"composition {metrics['k_composition_score']:.3f} | "
            f"epochs {int(metrics['epochs_run'])}"
        )
        for name, ok in checks.items():
            print(f"    {'PASS' if ok else 'FAIL'}  {name}")
    return m


m_emergence = report("results/induction_content_emergence.json", "emergence (early-stopped)")
m_full = report("results/induction_content.json", "full budget (3000 epochs)")

verdicts = [verdict(x)[0] for x in m_full["per_seed_metrics"]]
n_positive = verdicts.count("POSITIVE")
print(f"\nper-seed verdicts: {verdicts}")
if n_positive >= 2:
    print("RUN VERDICT: POSITIVE — a content-based induction head forms and the causal")
    print("interventions land. The word 'induction head' is now earned on this task.")
elif any(v == "PARTIAL" for v in verdicts):
    print("RUN VERDICT: PARTIAL — a copy head forms, but the causal evidence does not")
    print("support the induction claim. Report the missing criterion, do not re-run.")
else:
    print("RUN VERDICT: NEGATIVE — no content-based induction head at this budget.")
    print("Report it as a negative. The budget is not extended after the fact.

In [ ]:
# Rung 4 analyses the SAME model Rung 1 trained: the checkpoint is loaded,
# not retrained. That is what makes the cascade one model instead of two, and
# --source-task variable makes it analyse variable-distance data and measure
# the interventions at a copy query.
FOUND_HEAD = any(int(x["total_induction_heads"]) > 0 for x in m_full["per_seed_metrics"])
if FOUND_HEAD:
    !python -m src.experiments.exp4_circuit_patching --from-checkpoint checkpoints/induction_content --source-vocab-size 2048 --source-seq-len 64 --source-d-model 64 --source-n-layers 2 --source-n-heads 4 --source-task variable --seeds $SEEDS --manifest-path results/exp4_content.json
else:
    print("No head confirmed at the full budget — Rung 4 has no target and is skipped.")

In [ ]:
import glob
import json
import os


def first_head_seed(manifest):
    if not os.path.exists(manifest):
        return None
    m = json.load(open(manifest))
    for seed, metrics in zip(m["seeds"], m["per_seed_metrics"]):
        if int(metrics["total_induction_heads"]) > 0:
            return seed
    return None


if FOUND_HEAD:
    for seed, metrics in zip(m_full["seeds"], m_full["per_seed_metrics"]):
        print(f"exp4 seed {seed}: heads={int(metrics['total_induction_heads'])} | "
              f"ablation {metrics['mean_ablation_effect']:.3f} | "
              f"path patch {metrics['mean_path_patch_effect']:.3f} | "
              f"activation recovery {metrics['mean_activation_recovery']:.3f} | "
              f"checkpoint epoch {int(metrics['checkpoint_epoch'])}")

# Step 4 — the SAE on real activations from a checkpoint with a confirmed head.
seed = first_head_seed("results/induction_content.json")
ckpt_dir = "checkpoints/induction_content"
if seed is None:
    seed = first_head_seed("results/induction_content_emergence.json")
    ckpt_dir = "checkpoints/induction_content_emergence"

if seed is None:
    print("No seed confirmed a head — the exp5 real-activation run has no")
    print("confirmed-head checkpoint to harvest from. It is skipped, not faked.")
else:
    ckpt = glob.glob(f"{ckpt_dir}/*seed{seed}*.pt")
    print("harvesting from", ckpt)
    !python -m src.experiments.exp5_sae_dashboard --activations-from {ckpt[0]} --source-vocab-size 2048 --source-seq-len 64 --source-d-model 64 --source-n-layers 2 --source-n-heads 4 --source-task variable --seeds $SEEDS --manifest-path results/exp5_content.json

In [ ]:
# Collect every manifest this session produced, then download.
import glob
import os

manifests = sorted(
    glob.glob("results/induction_content*.json")
    + glob.glob("results/exp4_content.json")
    + glob.glob("results/exp5_content.json")
)
manifests = [p for p in manifests if "probe_" not in p]
print("manifests:", manifests)
!zip -j induction_content_results.zip $manifests
try:
    from google.colab import files

    files.download("induction_content_results.zip")
except ImportError:
    print("Kaggle: the zip is in the notebook output (/kaggle/working).")

## After downloading

1. Unzip into a scratch dir, then pin: `python scripts/pin_remote_run.py --from <dir>` — it validates provenance (git sha must exist locally, clean tree, seeds/aggregate contract) and installs only what passes.
2. `make verify-claims` must stay green — the new manifests need their `<!-- manifest: results/... -->` tags next to the numbers they back.
3. Report exactly what happened, including "no content-based head at the full budget": the answer to issue #200 is data, either way. The pre-registration says a negative is a negative.
4. Update the Rung 1 section of `portfolio/RESULTS.md` with the new numbers, per seed, and with the verdict the criteria produced.
5. If the verdict is POSITIVE, the Rung 4 and Rung 5 sections have to say which *kind* of head they were measured on: the cascade they ran on 2026-10-20 was measured on a prefix-copying head, and a content head is a different circuit.
